# Omnibus — Marey diagram + bus bunching

Two operational views that the per-stop aggregates can't show, both built from columns we already have (`distance_cum_m`, `stop_seq`, the timestamp set):

1. **Marey / time–distance diagram** — the canonical transit chart. Time on x, distance-along-line on y, one diagonal per trip. Slope = speed (flat = dwelling), converging diagonals = **bunching**, the gap to the planned line = delay. Everything about a line's day on one plot.
2. **Headway regularity** — at a busy stop, the gap between consecutive buses (actual vs planned). Surfaces the "two buses nose-to-tail, then a long gap" pathology that wrecks perceived reliability even when *average* delay looks fine.

Caveats as everywhere: Line-1 overlap window dropped, `|delay_arr_s| < 7200`, productive arrivals only.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
import numpy as np

df = pl.read_parquet("../data/parquet/features.parquet")
clean = df.filter((pl.col("source_window") != "Daten_Linie_1_2024-09_2025-08")
                  & (pl.col("delay_arr_s").abs() < 7200))

# --- pick a line / direction / day with dense service ---
LINE, DIRECTION, WIN = "6", "B", "06.10.2024_19.10.2024_ITCS"
DAY = "2024-10-11"   # a Friday with ~92 trips

sel = (clean.filter((pl.col("line") == LINE) & (pl.col("direction_label") == DIRECTION)
                    & (pl.col("source_window") == WIN) & pl.col("productive_arr")
                    & (pl.col("operating_day").cast(pl.Utf8) == DAY)
                    & pl.col("ts_arrival_actual_door").is_not_null())
       .sort("trip_id", "stop_seq"))
print(f"Line {LINE} dir {DIRECTION} on {DAY}: "
      f"{sel['trip_id'].n_unique()} trips, {sel.height} stop-events")

## 1 — Marey diagram

Each line is one bus's run: x = actual door-arrival time, y = metres travelled along the route. Diagonals are colored by the trip's worst delay — a diagonal that drifts red and starts *converging* with the one behind it is a bunching event forming.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
norm = Normalize(vmin=0, vmax=300)
cmap = plt.cm.RdYlGn_r

trip_maxdelay = {}
for tid, g in sel.group_by("trip_id", maintain_order=True):
    g = g.sort("stop_seq")
    t = g["ts_arrival_actual_door"].to_list()
    y = g["distance_cum_m"].to_numpy() / 1000.0   # km
    md_ = max(0, int(g["delay_arr_s"].max()))
    trip_maxdelay[tid] = md_
    ax.plot(t, y, "-", color=cmap(norm(md_)), linewidth=1.0, alpha=0.8)

ax.set_xlabel("time of day"); ax.set_ylabel("distance along route [km]")
ax.set_title(f"Marey diagram — Line {LINE}, direction {DIRECTION}, {DAY}\n"
             "each diagonal = one bus; color = worst delay on that run; "
             "converging lines = bunching")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.xaxis.set_major_locator(mdates.HourLocator(interval=1))
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
plt.colorbar(sm, ax=ax, label="worst arrival delay on the run [s]")
ax.grid(True, alpha=0.25)
fig.tight_layout(); plt.show()

**How to read it.** Steeper diagonal = faster bus; a flat spot = a long dwell or a hold. The vertical distance between two adjacent diagonals is the **headway** — when two lines pinch toward each other, the buses are bunching (the trailing bus catches the leader, then both crawl together while a gap opens behind). Watch the colour: bunching almost always coincides with the trailing run going red. The PM peak band is where the diagonals crowd and redden together — that's where an operator would insert a hold or short-turn.

## 2 — Headway regularity at the busiest stop

Pick the highest-traffic stop on this line+direction across the whole window. For each day, sort the actual arrivals and take consecutive gaps = **actual headway**; same on the planned times = **scheduled headway**. The standard bunching metric is the headway coefficient of variation (CoV = σ/μ; > ~0.5 = irregular) — but it must be computed **per hour**, because RVV's scheduled frequency itself changes through the day (every ~12 min at peak, every ~30 min at night). A whole-day CoV is dominated by that planned variation and hides the bunching; within an hour the schedule is ~constant, so an elevated *actual* CoV there is genuine bunching.

In [ ]:
line_dir = (clean.filter((pl.col("line") == LINE) & (pl.col("direction_label") == DIRECTION)
                         & (pl.col("source_window") == WIN) & pl.col("productive_arr")
                         # headway regularity is a through-stop metric: at a terminus the
                         # gaps are layover/dispatch spacing, not rider-felt bunching. Guard
                         # the "busiest stop" pick so changing LINE can't land on a terminus.
                         & ~pl.col("is_terminus")))
busiest = (line_dir.filter(pl.col("ts_arrival_actual_door").is_not_null())
           .group_by("stop_name").len().sort("len", descending=True).row(0, named=True)["stop_name"])

at_stop = (line_dir.filter((pl.col("stop_name") == busiest)
                          & pl.col("ts_arrival_actual_door").is_not_null()
                          & pl.col("ts_arrival_planned").is_not_null())
           .with_columns(day=pl.col("operating_day").cast(pl.Utf8)))

# consecutive headways within each day, tagged by arrival hour, in minutes
def headways(ts_col):
    out = []
    for _, g in at_stop.group_by("day"):
        s = sorted(g[ts_col].to_list())
        for a, b in zip(s, s[1:]):
            mins = (b - a).total_seconds() / 60
            if 0 < mins < 60:                 # ignore overnight gaps
                out.append((a.hour, a.hour + a.minute / 60, mins))
    return out

act = headways("ts_arrival_actual_door")
plan = headways("ts_arrival_planned")

def cov_by_hour(rows):
    out = {}
    for h in range(24):
        v = np.array([m for hr, _, m in rows if hr == h])
        if len(v) >= 10:
            out[h] = v.std() / v.mean()
    return out
cov_a, cov_p = cov_by_hour(act), cov_by_hour(plan)

fig, (axA, axB) = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [3, 2]})
axA.scatter([t for _, t, _ in plan], [m for _, _, m in plan], s=10,
            color="#2c7fb8", alpha=0.35, label="scheduled headway")
axA.scatter([t for _, t, _ in act], [m for _, _, m in act], s=12,
            color="#d73027", alpha=0.55, label="actual headway")
axA.set_xlabel("time of day [h]"); axA.set_ylabel("headway to previous bus [min]")
axA.set_title(f"Headways at '{busiest}' — Line {LINE} dir {DIRECTION}\n"
              "points near 0 = bunched buses; tall points = service gaps")
axA.legend(); axA.grid(True, alpha=0.25); axA.set_xlim(5, 23)

hrs = sorted(cov_a)
x = np.arange(len(hrs))
axB.bar(x - 0.2, [cov_p.get(h, 0) for h in hrs], width=0.4, color="#2c7fb8", label="scheduled")
axB.bar(x + 0.2, [cov_a[h] for h in hrs], width=0.4, color="#d73027", label="actual")
axB.axhline(0.5, color="#444", ls="--", lw=1)
axB.text(len(hrs) - 0.5, 0.52, "CoV 0.5 = bunching threshold", ha="right", fontsize=8, color="#444")
axB.set_xticks(x); axB.set_xticklabels([f"{h:02d}" for h in hrs], fontsize=7)
axB.set_xlabel("hour of day"); axB.set_ylabel("headway CoV (per hour)")
axB.set_title("Headway irregularity by hour")
axB.legend(); axB.grid(True, axis="y", alpha=0.25)
fig.tight_layout(); plt.show()

peak = max(cov_a, key=cov_a.get)
print(f"worst hour: {peak:02d}:00  actual CoV {cov_a[peak]:.2f} vs scheduled {cov_p.get(peak, float('nan')):.2f}")
print("evening (regular 30-min service) actual CoV: "
      + ", ".join(f"{h:02d}h={cov_a[h]:.2f}" for h in hrs if h >= 21))

**How to read it.** Left: the actual headways (red) smear toward both 0 (buses bunched nose-to-tail) and the high end (the gap that bunching leaves behind), while the schedule (blue) steps cleanly between its peak and off-peak frequencies. Right is the honest metric — CoV *per hour*, so the schedule's own frequency changes don't pollute it. The result is sharp: in the **morning peak (07–08h) actual CoV jumps to ~0.6**, well past the 0.5 bunching line, while **late evening sits near 0.0** (a bus every 30 min, and they actually arrive every 30 min). So the irregularity isn't random — it's concentrated exactly in the peak, when the most riders are waiting. That's a rider-felt failure mode distinct from raw delay, and a specific pitch line: "in the AM peak the timetable promises a bus every ~12 min; you actually get clumps of two then a long gap." (Naïvely CoV'ing the whole day gives ~0.5 for *both* actual and scheduled and hides this entirely — see why the per-hour split matters.)